# From Pre-Genesis to MSS-N: Inscriptions, Rewriting, Preservation, and Homology

**AuthorityEffect:** None.  
This notebook provides an exact, reproducible walkthrough of the journey from the austere Pre-Genesis inscription substrate ($\Sigma_{\text{pre}} = \{<, >, a, b\}$), through model discovery trials and formal rewriting strategies, to exact finite preservation, quotient homology, and CHSH correlation bounds.

It formalizes:
1. **Pre-Genesis Inscription Grammar (`pre_genesis.ebnf`)**: Parsing, four epistemic states, and flat non-nesting constraints.
2. **Rewriting Strategies & Observational Equivalence**: Opaque vs. exact-inscription conventions, no-transition quiescence, and the Forced Separation Theorem.
3. **Pre-Genesis Trial Records & 4 Branches**: Epistemological constraints, pre-registered branches A–D, and raw control responses.
4. **The Eight Named Failure Modes & Replay Necessity**: Executable detectors for semantic overcompletion, premature relations, lost state, and history-loss under projection.
5. **Finite Preservation & Homology**: Fiber report-sufficiency, demand step tracking, boundary operators, integer torsion, triangle degree classes, free-word normal forms, and CHSH Bell mixtures.


In [1]:
from pathlib import Path
import sys
import json
from fractions import Fraction

# Dynamically locate repository root and bind code paths
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'code/preservation.py').is_file())
sys.path.insert(0, str(root / 'code'))
sys.path.insert(0, str(root / 'code/grammar'))

from check_grammars import Grammar
from preservation import (
    report_factor, demand_step, boundary, graph_homology_ranks,
    cyclic_representative, component_totals, triangle_h1_class,
    triangle_degree_class, triangle_coboundary, protocol,
    free_reduce, bell_pair_distribution, bell_mixture, chsh
)
print("Environment initialized with preservation and grammar engines.")


Environment initialized with preservation and grammar engines.


## 1. The Pre-Genesis Inscription Grammar ($\Sigma_{\text{pre}}$) and Four Epistemic States

The Pre-Genesis inscription regime is defined over an austere alphabet:
$$\Sigma_{\text{pre}} = \Sigma_{\text{boundary}} \cup \Sigma_{\text{payload}} = \{<, >\} \cup \{a, b\}$$

The formal grammar (`code/grammar/pre_genesis.ebnf`) keeps **four distinct epistemic states** strictly apart:
1. **Absent Input**: No communication at all; matched by no grammar start symbol.
2. **Empty Word ($\varepsilon$)**: The length-zero string `""`; refused fail-closed by all public start symbols.
3. **Ground Token (`<>`)**: A 2-byte balanced boundary container with empty payload; possesses zero internal coordinates.
4. **Ground Carrier ($S(0)$)**: The semantic stage object adopted by stage $S(0)$; a mathematical entity rather than a wire string.

Additionally, the substrate enforces:
- **Flat Inscription Algebra**: Nesting depth is strictly $\operatorname{depth}(t) = 1$. Nested tokens like `<<>>` or `<<a>>` are invalid syntax and refused fail-closed.
- **Direct Adjacency**: Inscriptions concatenate directly without ambient formatting: `<><a><ab>`.
- **Regime Enclosures**: Multi-token collections are enclosed in braces `{...}`, where `{}` is the quiescent empty regime (distinct from $\varepsilon$).


In [2]:
# Load the official Pre-Genesis EBNF specification
grammar_source = (root / "code/grammar/pre_genesis.ebnf").read_text(encoding="utf-8")
pre_grammar = Grammar(grammar_source)

# 1. Verify Ground Token
assert pre_grammar.accepts("ground_token", "<>")
assert not pre_grammar.accepts("ground_token", "")
assert not pre_grammar.accepts("ground_token", "<a>")

# 2. Verify Occupied Tokens and Flat Algebra (Depth = 1)
assert pre_grammar.accepts("occupied_token", "<a>")
assert pre_grammar.accepts("occupied_token", "<b>")
assert pre_grammar.accepts("occupied_token", "<ab>")
assert pre_grammar.accepts("occupied_token", "<ba>")
assert pre_grammar.accepts("occupied_token", "<abba>")

# Nesting depth > 1 is strictly refused fail-closed
assert not pre_grammar.accepts("inscription", "<<>>")
assert not pre_grammar.accepts("inscription", "<<a>>")
assert not pre_grammar.accepts("inscription", "<a<b>>")

# 3. Verify Empty Word refusal across all public starts
assert not pre_grammar.accepts("ground_token", "")
assert not pre_grammar.accepts("occupied_token", "")
assert not pre_grammar.accepts("inscription", "")
assert not pre_grammar.accepts("pre_genesis_regime", "")

# 4. Verify Regime Enclosures
assert pre_grammar.accepts("pre_genesis_regime", "{}")  # Empty regime
assert pre_grammar.accepts("pre_genesis_regime", "{<>}")
assert pre_grammar.accepts("pre_genesis_regime", "{<a>,<ab>}")
assert not pre_grammar.accepts("pre_genesis_regime", "{<x>}")  # 'x' not in Sigma_pre

print({
    "alphabet": ["<", ">", "a", "b"],
    "ground_token_valid": pre_grammar.accepts("ground_token", "<>"),
    "empty_word_refused": not pre_grammar.accepts("inscription", ""),
    "nested_boundary_refused": not pre_grammar.accepts("inscription", "<<>>"),
    "empty_regime_accepted": pre_grammar.accepts("pre_genesis_regime", "{}"),
    "two_token_regime_accepted": pre_grammar.accepts("pre_genesis_regime", "{<a>,<ab>}")
})


{'alphabet': ['<', '>', 'a', 'b'], 'ground_token_valid': True, 'empty_word_refused': True, 'nested_boundary_refused': True, 'empty_regime_accepted': True, 'two_token_regime_accepted': True}


## 2. Observational Equivalence, Conventions, and No-Transition Quiescence

When a new token (e.g. `<a>`) is supplied after `<>`, what has changed?
The answer depends on the convention chosen:
- **Opaque Convention**: State identity is quotiented by the active observation set $O$. If $O = \emptyset$, the regime can observe no internal distinctions, so $\langle\rangle \approx_\emptyset \langle a\rangle$.
- **Exact-Inscription Convention**: State identity is wire string equality over $\Sigma_{\text{pre}}^*$. Because `"<a>"` $\neq$ `"<>"` in $\Sigma_{\text{pre}}^*$, they are distinct states.

Furthermore:
- **No-Transition Quiescence**: Prior to Genesis event admission, the rewrite relation is empty ($\mathcal{R} = \emptyset$). Every well-formed inscription is irreducible; no autonomous step is forced.
- **Forced Separation Theorem**: If an external task requires distinguishing $x$ and $y$ when $x \approx_O y$, the regime is forced to introduce a new observation predicate $o^* \in \mathcal{O}$. Crucially, the lift is **non-unique** (e.g., length predicate vs. letter-content predicate).


In [3]:
def observational_equivalence(token_x: str, token_y: str, predicates: list) -> bool:
    """Two tokens are equivalent iff all available observation predicates agree."""
    return all(pred(token_x) == pred(token_y) for pred in predicates)

# Opaque Convention: empty observation set O = []
O_empty = []
assert observational_equivalence("<>", "<a>", O_empty) == True
assert observational_equivalence("<a>", "<ab>", O_empty) == True

# Exact-Inscription Convention: string equality over Sigma_pre*
assert ("<>" == "<a>") == False
assert ("<a>" == "<ab>") == False

# No-Transition Quiescence: R = {}
rewrite_rules = {}  # Empty transition relation
def step(token: str, rules: dict) -> str | None:
    return rules.get(token, None)

assert step("<>", rewrite_rules) is None
assert step("<a>", rewrite_rules) is None

# Forced Separation Theorem: Non-unique lift
# Task requires separating '<a>' and '<ab>'
pred_length = lambda t: len(t.strip("<>"))
pred_has_b = lambda t: "b" in t.strip("<>")

# Lift 1: payload length
assert pred_length("<a>") != pred_length("<ab>")
# Lift 2: letter 'b' detector
assert pred_has_b("<a>") != pred_has_b("<ab>")

print({
    "opaque_equivalence_under_empty_O": observational_equivalence("<>", "<a>", O_empty),
    "exact_inscription_inequality": "<>" != "<a>",
    "quiescence_no_autonomous_step": step("<>", rewrite_rules) is None,
    "forced_separation_lift_length": (pred_length("<a>"), pred_length("<ab>")),
    "forced_separation_lift_has_b": (pred_has_b("<a>"), pred_has_b("<ab>"))
})


{'opaque_equivalence_under_empty_O': True, 'exact_inscription_inequality': True, 'quiescence_no_autonomous_step': True, 'forced_separation_lift_length': (1, 2), 'forced_separation_lift_has_b': (False, True)}


## 3. Pre-Genesis Experiment Design and the Four Pre-Registered Branches

The Pre-Genesis experiments investigated:
> *When a frontier language model is given an uninterpreted finite inscription and is required to distinguish external presentation from internal expressibility, what symbolic resources does it introduce, in what order, and under what failures does it enlarge the regime?*

Core design principles (`01_experiment_design.md`):
- **Constrain epistemology, not mathematics**: The prompt specifies what may be assumed, but never reveals comparison targets (e.g. that `<>` represents Primitive Difference or that `<a>` is the expected next state).
- **Four Pre-Registered Branches**:
  - **Branch A**: Proposes `<a>` (follow symbol, ask what was added).
  - **Branch B**: Proposes another payload like `<x>` (follow exactly).
  - **Branch C**: Proposes a relation, duplication, nesting, or operator.
  - **Branch D**: Declares that **no continuation is forced** (counted as a primary success!).


In [4]:
def classify_pre_genesis_continuation(proposal: str | None) -> str:
    """Classifies a model continuation into pre-registered branches A, B, C, D."""
    if proposal is None or "no" in proposal.lower() or "none" in proposal.lower():
        return "Branch D: No continuation forced (Autonomous Quiescence Confirmed)"
    clean = proposal.strip()
    if clean == "<a>":
        return "Branch A: Canonical payload 'a' proposed"
    if clean.startswith("<") and clean.endswith(">") and len(clean) == 3 and clean[1] not in "ab":
        return f"Branch B: Novel external payload '{clean[1]}' proposed"
    if clean == "<>" or " " in clean or clean in {"<<>>", "<->", "rule"}:
        return "Branch C: Relation, repetition, or operator proposed"
    return "Branch C: Structured extension proposed"

# Evaluate trial responses from historical trial records:
t1_reply = "None. With zero declared rules, nothing compels or licenses any next inscription."
t2_reply = "Next inscription: <> (assuming a rule of repetition)"
fr_reply = "Inscribe adjacent sequence <> <>"

print({
    "Trial_1_Classification": classify_pre_genesis_continuation(t1_reply),
    "Trial_2_Classification": classify_pre_genesis_continuation(t2_reply),
    "Full_Replay_Classification": classify_pre_genesis_continuation(fr_reply),
    "Direct_a_Classification": classify_pre_genesis_continuation("<a>"),
    "External_x_Classification": classify_pre_genesis_continuation("<x>")
})


{'Trial_1_Classification': 'Branch D: No continuation forced (Autonomous Quiescence Confirmed)', 'Trial_2_Classification': 'Branch C: Relation, repetition, or operator proposed', 'Full_Replay_Classification': 'Branch C: Relation, repetition, or operator proposed', 'Direct_a_Classification': "Branch A: Canonical payload 'a' proposed", 'External_x_Classification': "Branch B: Novel external payload 'x' proposed"}


## 4. The Eight Named Failure Modes and Replay Necessity

The Pre-Genesis trial series identified eight concrete failure modes (`04_named_failure_modes.md`):
1. **Semantic Overcompletion**: Mapping `<>` to external domain semantics (SQL `<>`, generics, bra-ket, HTML).
2. **Premature Relation**: Claiming `<a>` and `<ab>` are "identified" when no equivalence relation was declared.
3. **Metalanguage Mistaken for Import**: Flagging prompt meta-words ("we", "rule") as internal wire symbols.
4. **Accepted False Provenance**: Answering queries about claims the model never made.
5. **Lost State**: Forgetting prior symbols (`<a>`) and asserting the regime has only `<>`.
6. **Stale Certification**: Certifying a step without evaluating the history that produced it.
7. **Local Green, Global Red**: Each step is locally consistent with its premise, but the initial premise was false for the actual history.
8. **Bundled Extension**: Concealing multiple undeclared capabilities (branching + matching + rewriting) inside a "one-step" move.

### Replay Necessity Theorem
If an auditor evaluates an execution through a lossy projection $\pi: \mathcal{H} \to \mathcal{H}_{\text{partial}}$ that drops history, there exist distinct histories $h_1 \neq h_2$ with $\pi(h_1) = \pi(h_2)$ but differing ground truth $\sigma(h_1) \neq \sigma(h_2)$. Consequently, fail-closed verification requires **append-only history retention**.


In [5]:
# 1. Detector for Premature Relation
def check_premature_relation(relation_claimed: bool, declared_rules: set) -> bool:
    if relation_claimed and "equivalence_rule" not in declared_rules:
        return True  # Failure mode: Premature Relation
    return False

assert check_premature_relation(relation_claimed=True, declared_rules=set()) == True
assert check_premature_relation(relation_claimed=True, declared_rules={"equivalence_rule"}) == False

# 2. Detector for Lost State
def check_lost_state(history: list[str], current_inventory: set[str]) -> set[str]:
    dropped = set(history) - current_inventory
    return dropped

history_demo = ["<>", "<a>"]
lossy_inventory = {"<>"}
assert check_lost_state(history_demo, lossy_inventory) == {"<a>"}

# 3. Demonstration of Replay Necessity: Lossy Projection Failure
def auditor_evaluate(history: list[str], lossy_projection: bool) -> dict:
    if lossy_projection:
        # Auditor only sees the last turn
        view = history[-1]
    else:
        # PFP requirement: full append-only history
        view = tuple(history)
    return {"auditor_view": view, "state_count": len(set(history))}

h1 = ["<>", "<a>", "<b>"]
h2 = ["<b>", "<b>"]

# Under lossy single-turn projection:
view1 = auditor_evaluate(h1, lossy_projection=True)["auditor_view"]
view2 = auditor_evaluate(h2, lossy_projection=True)["auditor_view"]
assert view1 == view2 == "<b>"  # Auditor cannot tell h1 from h2!

# Under full PFP history retention:
full1 = auditor_evaluate(h1, lossy_projection=False)["auditor_view"]
full2 = auditor_evaluate(h2, lossy_projection=False)["auditor_view"]
assert full1 != full2

print({
    "premature_relation_detected": check_premature_relation(True, set()),
    "lost_state_detected": sorted(check_lost_state(history_demo, lossy_inventory)),
    "lossy_auditor_ambiguity": (view1, view2),
    "replay_necessity_distinction": full1 != full2
})


{'premature_relation_detected': True, 'lost_state_detected': ['<a>'], 'lossy_auditor_ambiguity': ('<b>', '<b>'), 'replay_necessity_distinction': True}


## 5. Report Sufficiency and Fiber Invariance

Moving from Pre-Genesis to the Witnessed Difference Programme:
The **report-fiber criterion** states that an answer $s \in S$ can be recovered from a reported observation $r \in R$ if and only if the answer is constant on every report fiber. If any fiber contains multiple answers, the report loses information and the reconstruction factor is refused fail-closed.


In [6]:
# Sufficient report: constant on each report fiber
assert report_factor([0, 1, 0], ['a', 'b', 'a']) == {0: 'a', 1: 'b'}

# Information-losing report: fiber for report 0 contains both 'a' and 'b'
assert report_factor([0, 0], ['a', 'b']) is None

# Demand accounting: residual active demands after discharge and birth
active, discharged, births = {0, 1}, {0}, {2}
new_demands = demand_step(active, discharged, births)
assert len(new_demands) == len(active) - len(discharged) + len(births)

print({
    "sufficient_factor": report_factor([0, 1, 0], ['a', 'b', 'a']),
    "conflicting_report_refused": report_factor([0, 0], ['a', 'b']) is None,
    "active_demands": sorted(new_demands)
})


{'sufficient_factor': {0: 'a', 1: 'b'}, 'conflicting_report_refused': True, 'active_demands': [1, 2]}


## 6. Graph Boundaries and Cycle Homology Ranks

Topological cycles in finite simplicial networks:
An oriented cycle satisfies $\partial_{1}(c) = 0$. In a triangle graph ($K_3$), the cycle rank $H_1$ is 1. Attaching a 2-cell boundary fills the face and kills the cycle.


In [7]:
triangle = [(0, 1), (1, 2), (2, 0)]
# Boundary of uniform 1-chain is zero (it is a cycle)
assert boundary(3, triangle, [1, 1, 1]) == (0, 0, 0)
# Homology ranks (H0, H1) for triangle graph: 1 component, 1 cycle
assert graph_homology_ranks(3, triangle) == (1, 1)

print({
    "triangle_cycle_boundary": boundary(3, triangle, [1, 1, 1]),
    "homology_ranks_H0_H1": graph_homology_ranks(3, triangle)
})


{'triangle_cycle_boundary': (0, 0, 0), 'homology_ranks_H0_H1': (1, 1)}


## 7. Integral Classes Retain Torsion

For a chain complex with boundary $k \mapsto m \cdot k$ and zero next differential, the middle homology class of $n$ is its integer residue modulo $m$.
- Modulus zero retains the entire integer $\mathbb{Z}$.
- At modulus $m = 2$, the class of $1$ is nonzero ($1 \neq 0$), but its double is zero ($1 + 1 \equiv 0 \pmod 2$). This exhibits finite order-two torsion.


In [8]:
assert cyclic_representative(2, 1) == 1
assert cyclic_representative(2, 1 + 1) == 0
assert cyclic_representative(-2, -3) == 1
assert cyclic_representative(0, -3) == -3

print({
    "order_two_generator": cyclic_representative(2, 1),
    "order_two_double": cyclic_representative(2, 2),
    "zero_modulus_retained": cyclic_representative(0, -3)
})


{'order_two_generator': 1, 'order_two_double': 0, 'zero_modulus_retained': -3}


## 8. Integral Component Coordinates

An integral 0-chain assigns one integer per vertex. Its class is invariant under edge boundary shifts $\partial_1(e) = v_{\text{dst}} - v_{\text{src}}$. Adding an edge boundary shifts coefficients between vertices in the same connected component, preserving the component sum.


In [9]:
edges = [(3, 2), (2, 0), (3, 2), (1, 1)]
chain = (2, -1, 3, -2)
totals = component_totals(4, edges, chain)
assert totals == ((0, 3), (1, -1))

# Shift by an arbitrary edge boundary
change = boundary(4, edges, (2, -3, 1, 7))
shifted = tuple(a + b for a, b in zip(chain, change, strict=True))
assert component_totals(4, edges, shifted) == totals

print({
    "component_totals": totals,
    "boundary_shift_invariance": component_totals(4, edges, shifted) == totals
})


{'component_totals': ((0, 3), (1, -1)), 'boundary_shift_invariance': True}


## 9. Actual Triangle Classes and Degree Attachments

The integral triangle cycle space is generated by diagonal triples $(k, k, k)$.
- Unfilled: quotient is $\mathbb{Z}$.
- Filled with degree $m$ attachment: quotient is $\mathbb{Z}/(m\mathbb{Z})$.
- Non-cycle inputs are refused fail-closed.


In [10]:
assert triangle_h1_class(False, (3, 3, 3)) == 3
assert triangle_h1_class(True, (3, 3, 3)) == 0
assert triangle_degree_class(2, (1, 1, 1)) == 1
assert triangle_degree_class(2, (2, 2, 2)) == 0
assert triangle_degree_class(0, (-3, -3, -3)) == -3
assert protocol('v1\ttriangle_degree\t2,1,1,0') == 'REFUSE'
assert triangle_coboundary((2, -5, 3)) == 0

print({
    "unfilled_class": triangle_h1_class(False, (3, 3, 3)),
    "filled_class": triangle_h1_class(True, (3, 3, 3)),
    "degree_two_generator": triangle_degree_class(2, (1, 1, 1)),
    "degree_two_double": triangle_degree_class(2, (2, 2, 2)),
    "malformed_refused": protocol('v1\ttriangle_degree\t2,1,1,0')
})


{'unfilled_class': 3, 'filled_class': 0, 'degree_two_generator': 1, 'degree_two_double': 0, 'malformed_refused': 'REFUSE'}


## 10. Free-Word Normal Form and the Commutator

In the free group on $\{a, b\}$, capital letters denote inverse generators ($A = a^{-1}$, $B = b^{-1}$).
- Reduced words classify equality in the free group.
- The commutator $abAB$ has zero net exponent sums for both generators, yet is a non-empty reduced word. It represents a non-trivial loop in the fundamental group.


In [11]:
assert free_reduce('abBA') == ''
assert free_reduce('abAB') == 'abAB'
assert free_reduce('ab') != free_reduce('ba')

commutator = free_reduce('abAB')
assert commutator.count('a') - commutator.count('A') == 0
assert commutator.count('b') - commutator.count('B') == 0

print({
    "trivial_reduction": free_reduce('abBA'),
    "commutator_normal_form": commutator,
    "abelian_exponent_sums": (
        commutator.count('a') - commutator.count('A'),
        commutator.count('b') - commutator.count('B')
    )
})


{'trivial_reduction': '', 'commutator_normal_form': 'abAB', 'abelian_exponent_sums': (0, 0)}


## 11. Pair Distributions versus Joint CHSH Mixtures

In the Bell-CHSH framework:
- The pairwise target has uniform marginals and maximal correlation sum $S = 4$.
- Every normalized nonnegative rational mixture of the sixteen deterministic joint assignments satisfies the classical Bell-CHSH bound:
$$|S_{\text{joint}}| \le 2$$
The rational evaluator confirms this bound exhaustively across all deterministic extremal states.


In [12]:
# Pair distributions with exact rational probabilities
for negative in (False, True):
    pair = bell_pair_distribution(negative)
    assert sum(pair) == 1
    assert pair[0] + pair[1] == pair[2] + pair[3] == Fraction(1, 2)

# Uniform mixture of all 16 deterministic assignments yields score 0
assert bell_mixture([1] * 16) == (0, 0, 0, 0, 0)

# Check all 16 extremal deterministic vertices satisfy |S| <= 2
for index in range(16):
    masses = [int(i == index) for i in range(16)]
    score = bell_mixture(masses)[4]
    assert abs(score) <= 2

print({
    "uniform_joint_score": bell_mixture([1] * 16)[4],
    "extremal_deterministic_max_score": max(abs(bell_mixture([int(i == k) for i in range(16)])[4]) for k in range(16)),
    "pairwise_target_score": chsh((1, 1, -1, 1))[1]
})


{'uniform_joint_score': Fraction(0, 1), 'extremal_deterministic_max_score': Fraction(2, 1), 'pairwise_target_score': 4}
